# 텍스트 파일 로더

일반 텍스트(`.txt`) 파일은 가장 단순하지만 가장 범용적인 문서 형식입니다.

이 노트북에서는 `TextLoader`를 사용하여 텍스트 파일을 Document 객체로 변환하고, 다양한 인코딩을 처리하는 방법을 학습합니다.

**학습 내용**:
- 기본 텍스트 파일 로딩
- 인코딩 지정 및 자동 감지
- 여러 인코딩 파일 일괄 처리
- DirectoryLoader와의 통합


# 08. 텍스트 파일 로더

## 학습 목표
- `TextLoader`로 `.txt` 파일을 Document로 변환해요
- 인코딩 문제(utf-8, cp949, euc-kr)를 해결하는 방법을 익혀요
- `DirectoryLoader`와 결합해서 여러 파일을 일괄 로딩해요

## 사전 지식
- 00-Document-Loader 노트북 완료
- 한글 인코딩 개념 (utf-8, cp949) 기본 이해

---

> **RAG 파이프라인 위치**: Load 단계 — 가장 기본적인 텍스트 파일을 처리해요. 한국어 파일은 **인코딩 설정**이 핵심이에요.


> 🎯 **강의 포인트**: TXT 로더에서 가장 흔한 오류는 인코딩 문제입니다. 한국어 파일에서 `UnicodeDecodeError`가 발생하면 `encoding="cp949"` 또는 `autodetect_encoding=True`로 해결하세요.

In [1]:
from dotenv import load_dotenv
load_dotenv()

FILE_PATH = "./data/appendix-keywords.txt"


## 1. TextLoader 기본 사용

`TextLoader`는 텍스트 파일을 로드하는 가장 기본적인 로더입니다.

**주요 특징**:
- 간단한 사용법
- 단일 파일 → 단일 Document 변환
- 인코딩 지정 가능


In [2]:
# ============================================================
# TODO: TextLoader로 텍스트 파일을 로드해보세요
# 힌트: TextLoader(FILE_PATH, encoding="utf-8") → loader.load()
# 예상 결과: 1개의 Document가 생성되고 파일 경로가 메타데이터의 source에 포함됩니다
# ============================================================

from langchain_community.document_loaders import TextLoader

# 1단계: TextLoader 생성 (인코딩 지정)
# - encoding="utf-8": 한국어 파일은 utf-8 또는 cp949 지정 필요
loader = TextLoader(FILE_PATH, encoding="utf-8")

# 2단계: 파일 로드
docs = loader.load()

# 3단계: 결과 확인
print(f"로드된 문서 개수: {len(docs)}")
print(f"\n첫 번째 문서의 타입: {type(docs[0])}")
print("=" * 60)
print("📄 텍스트 정보:")
print("=" * 60)
print(f"전체 텍스트 길이: {len(docs[0].page_content)} 글자")
print(f"\n내용 미리보기 (처음 300자):")
print(docs[0].page_content[:300])
print("\n" + "=" * 60)
print("🏷️  메타데이터:")
print("=" * 60)
print(docs[0].metadata)

로드된 문서 개수: 1

첫 번째 문서의 타입: <class 'langchain_core.documents.base.Document'>
📄 텍스트 정보:
전체 텍스트 길이: 5733 글자

내용 미리보기 (처음 300자):
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단

🏷️  메타데이터:
{'source': './data/appendix-keywords.txt'}


> 🔑 **핵심 개념**: 인코딩 선택 기준 — **utf-8**: 최신 시스템/웹 문서 | **cp949**: Windows 메모장에서 작성한 한글 파일 | **euc-kr**: 레거시 시스템 | **autodetect**: 인코딩을 모를 때

## 2. 인코딩 처리

텍스트 파일은 다양한 인코딩 방식으로 저장될 수 있습니다.

한국어 텍스트의 경우 특히 인코딩 문제가 자주 발생하므로, 올바른 인코딩을 지정하는 것이 중요합니다.

### 주요 인코딩 방식

| 인코딩 | 설명 | 사용 예 |
|-------|------|---------|
| `utf-8` | 유니코드 표준, 가장 권장 | 최신 시스템, 웹 문서 |
| `cp949` | Windows 한글 인코딩 | Windows 메모장 (구버전) |
| `euc-kr` | 한글 완성형 인코딩 | 레거시 시스템 |
| `latin-1` | 서유럽 문자 | 영문 레거시 파일 |


In [3]:
# 인코딩을 명시적으로 지정하는 예제
encodings_to_try = ["utf-8", "cp949", "euc-kr"]

for encoding in encodings_to_try:
    try:
        loader = TextLoader(FILE_PATH, encoding=encoding)
        docs = loader.load()
        print(f"✅ {encoding}: 성공 (텍스트 길이: {len(docs[0].page_content)} 글자)")
        break
    except Exception as e:
        print(f"❌ {encoding}: 실패 - {str(e)[:50]}...")


✅ utf-8: 성공 (텍스트 길이: 5733 글자)


## 3. 인코딩 자동 감지

인코딩을 모르는 경우 `autodetect_encoding=True` 옵션을 사용하면 자동으로 인코딩을 감지합니다.

이 기능은 다양한 인코딩의 파일을 처리할 때 매우 유용합니다.


In [4]:
# ============================================================
# TODO: autodetect_encoding=True로 인코딩을 자동 감지해보세요
# 힌트: TextLoader(FILE_PATH, autodetect_encoding=True)
# 예상 결과: 인코딩을 지정하지 않아도 파일이 성공적으로 로드됩니다
# ============================================================

# 1단계: 인코딩 자동 감지 옵션 사용
# - autodetect_encoding=True: chardet 라이브러리로 인코딩 자동 탐지
try:
    loader = TextLoader(FILE_PATH, autodetect_encoding=True)
    docs = loader.load()
    print("✅ 인코딩 자동 감지 성공")
    print(f"텍스트 길이: {len(docs[0].page_content)} 글자")
    print(f"\n처음 200자:")
    print(docs[0].page_content[:200])
except Exception as e:
    print(f"⚠️ 오류: {e}")
    print("💡 encoding 파라미터를 명시적으로 지정하세요")

✅ 인코딩 자동 감지 성공
텍스트 길이: 5733 글자

처음 200자:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정


> 💡 **실무 팁**: `silent_errors=True`는 운영 환경에서 필수입니다. 수백 개 파일 중 인코딩이 깨진 파일 하나 때문에 전체 작업이 중단되는 것을 방지합니다. 단, 나중에 실패한 파일 목록을 별도로 기록해두세요.

## 4. DirectoryLoader와 함께 사용

여러 텍스트 파일을 한 번에 로드할 때는 `DirectoryLoader`와 함께 사용합니다.

**핵심 옵션**:
- `autodetect_encoding=True`: 각 파일의 인코딩을 자동 감지
- `silent_errors=True`: 로드 실패 파일을 건너뛰고 계속 진행

**시나리오**: `data/` 디렉토리의 모든 `.txt` 파일을 일괄 로드


In [5]:
# ============================================================
# TODO: DirectoryLoader로 data 폴더의 모든 .txt 파일을 로드해보세요
# 힌트: DirectoryLoader("./data", glob="**/*.txt", loader_cls=TextLoader, silent_errors=True)
# 예상 결과: 여러 .txt 파일이 로드되고 각 파일의 경로가 메타데이터에 포함됩니다
# ============================================================

from langchain_community.document_loaders import DirectoryLoader

# 1단계: TextLoader 옵션 설정 (인코딩 자동 감지)
text_loader_kwargs = {"autodetect_encoding": True}

# 2단계: DirectoryLoader 생성
# - glob="**/*.txt": 모든 하위 디렉토리의 .txt 파일
# - silent_errors=True: 오류 발생 시 건너뛰기 (전체 작업 중단 방지)
loader = DirectoryLoader(
    "./data",
    glob="**/*.txt",
    loader_cls=TextLoader,
    silent_errors=True,
    loader_kwargs=text_loader_kwargs,
)

# 3단계: 일괄 로드
docs = loader.load()

print(f"로드된 파일 수: {len(docs)}")
print("\n" + "=" * 60)
print("📁 로드된 파일 목록:")
print("=" * 60)
for i, doc in enumerate(docs):
    source = doc.metadata.get("source", "Unknown")
    length = len(doc.page_content)
    print(f"{i+1}. {source} ({length} 글자)")

Error loading file data/appendix-keywords-CP949.txt: No module named 'chardet'
Error loading file data/appendix-keywords-EUCKR.txt: No module named 'chardet'


로드된 파일 수: 4

📁 로드된 파일 목록:
1. data/reference.txt (379 글자)
2. data/chain-of-density.txt (2404 글자)
3. data/appendix-keywords.txt (5733 글자)
4. data/appendix-keywords-utf8.txt (5733 글자)


## 핵심 정리 및 마무리

### 인코딩 선택 가이드

| 상황 | 권장 인코딩 |
|------|-----------|
| 최신 시스템, 웹 문서 | `utf-8` |
| Windows 메모장으로 작성한 한글 파일 | `cp949` |
| 오래된 한글 시스템 | `euc-kr` |
| 인코딩을 모를 때 | `autodetect_encoding=True` |

### 실무 팁
> `UnicodeDecodeError`가 발생하면 당황하지 말고 `autodetect_encoding=True`를 먼저 시도해 보세요. 그래도 안 되면 `cp949`를 명시적으로 지정하는 게 한국어 파일에서 대부분 해결돼요.

---

## 다음 예고

다음은 구조화된 데이터 형식인 JSON 파일을 다뤄볼게요.

- **09-JSON-Loader**: `jq_schema`로 JSON 특정 필드를 page_content로 추출
